# Experimento de Carga — Notebook Interativo

Este notebook reproduz a jornada do script `run_experiment.py`, etapa a etapa, para que você possa debugar cada fase de forma isolada.

**Fluxo por rodada:**
1. Configuração de parâmetros
2. Leitura/validação das métricas Prometheus (PromQL)
3. Disparo do Locust (com warmup + execução)
4. Coleta das métricas no intervalo da rodada
5. Salvamento do CSV
6. Análise rápida dos resultados

---
## Etapa 1 — Imports e utilitários

In [ ]:
from __future__ import annotations

import csv
import datetime as dt
import json
import shlex
import subprocess
import time
import urllib.parse
import urllib.request
from pathlib import Path
from typing import Dict, List, Optional

print("Imports OK")

In [ ]:
def utc_iso_from_epoch(epoch_seconds: float) -> str:
    return dt.datetime.fromtimestamp(epoch_seconds, tz=dt.timezone.utc).isoformat()

def now_epoch() -> float:
    return time.time()

def _http_get_json(url: str, timeout: int) -> dict:
    req = urllib.request.Request(url, method="GET")
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def _parse_vector_or_scalar(payload: dict) -> Optional[float]:
    if payload.get("status") != "success":
        return None
    data = payload.get("data", {})
    result_type = data.get("resultType")
    result = data.get("result")
    if result_type == "vector":
        if not result:
            return None
        values: List[float] = []
        for item in result:
            try:
                values.append(float(item["value"][1]))
            except (TypeError, ValueError, IndexError, KeyError):
                continue
        return sum(values) if values else None
    if result_type == "scalar":
        try:
            return float(result[1])
        except (TypeError, ValueError, IndexError):
            return None
    return None

def query_prometheus_at(
    prom_url: str, promql: str, when_epoch: float, timeout: int
) -> Optional[float]:
    endpoint = prom_url.rstrip("/") + "/api/v1/query"
    query = urllib.parse.urlencode({"query": promql, "time": f"{when_epoch:.3f}"})
    url = endpoint + "?" + query
    try:
        payload = _http_get_json(url, timeout=timeout)
        return _parse_vector_or_scalar(payload)
    except Exception as e:
        print(f"  [WARN] query_prometheus_at falhou: {e}")
        return None

def query_prometheus_range(
    prom_url: str,
    promql: str,
    start_epoch: float,
    end_epoch: float,
    step_seconds: int,
    timeout: int,
) -> Dict[int, Optional[float]]:
    endpoint = prom_url.rstrip("/") + "/api/v1/query_range"
    params = {
        "query": promql,
        "start": f"{start_epoch:.3f}",
        "end": f"{end_epoch:.3f}",
        "step": str(step_seconds),
    }
    url = endpoint + "?" + urllib.parse.urlencode(params)
    try:
        payload = _http_get_json(url, timeout=timeout)
    except Exception as e:
        print(f"  [WARN] query_prometheus_range falhou: {e}")
        return {}

    if payload.get("status") != "success":
        return {}

    result_type = payload.get("data", {}).get("resultType")
    result = payload.get("data", {}).get("result", [])
    if result_type != "matrix" or not isinstance(result, list):
        return {}

    merged: Dict[int, float] = {}
    for series in result:
        for item in series.get("values", []):
            try:
                ts = int(float(item[0]))
                val = float(item[1])
            except (TypeError, ValueError, IndexError):
                continue
            merged[ts] = merged.get(ts, 0.0) + val

    out: Dict[int, Optional[float]] = {}
    for ts in range(int(start_epoch), int(end_epoch) + 1, step_seconds):
        out[ts] = merged.get(ts)
    return out

def state_for_timestamp(ts: int, disabled_end: float, warmup_end: float, run_end: float) -> str:
    if ts < int(disabled_end):
        return "DESATIVADA"
    if ts < int(warmup_end):
        return "AQUECIMENTO"
    if ts <= int(run_end):
        return "EXECUCAO"
    return "DESATIVADA"

print("Funções auxiliares definidas OK")

---
## Etapa 2 — Configuração dos parâmetros

Edite as variáveis abaixo antes de continuar.

In [ ]:
# ── Prometheus ────────────────────────────────────────────────────────────────
PROM_URL = "http://localhost:9090"
PROM_TIMEOUT_SECONDS = 10

# ── Experimento ──────────────────────────────────────────────────────────────
ROUNDS = 3
DISABLED_SECONDS = 0        # tempo idle antes do aquecimento
WARMUP_SECONDS = 60         # duração do aquecimento
RUN_SECONDS = 600           # duração da execução após aquecimento
SAMPLE_INTERVAL_SECONDS = 30

# ── Locust ───────────────────────────────────────────────────────────────────
LOCUST_CMD = "kubectl -n loadtest exec deploy/locust-web -- locust"
LOCUST_FILE = "/mnt/locust/locustfile.py"
LOCUST_HOST = "http://front-end.sock-shop.svc.cluster.local"
USERS = 50
SPAWN_RATE = 20.0
LOCUST_EXTRA_ARGS = ""
LOCUST_GRACE_SECONDS = 30
FAIL_ON_LOCUST_ERROR = False

# ── Saída ─────────────────────────────────────────────────────────────────────
OUTPUT_CSV = "../data/resultado_experimento.csv"

print("Parâmetros:")
print(f"  Prometheus: {PROM_URL}")
print(f"  Rodadas: {ROUNDS}")
print(f"  Aquecimento: {WARMUP_SECONDS}s | Execução: {RUN_SECONDS}s | Step: {SAMPLE_INTERVAL_SECONDS}s")
print(f"  Locust: {LOCUST_CMD}")
print(f"  Host: {LOCUST_HOST} | Usuários: {USERS} | Spawn-rate: {SPAWN_RATE}")
print(f"  Saída: {OUTPUT_CSV}")

---
## Etapa 3 — Definição / validação das métricas PromQL

In [ ]:
# Você pode substituir por um dict carregado de um arquivo JSON externo:
# import json; METRICS = json.loads(Path("metrics.json").read_text())

METRICS: Dict[str, str] = {
    "qps_total": 'sum(rate(request_duration_seconds_count{route!="metrics"}[1m]))',
    "qps_2xx":   'sum(rate(request_duration_seconds_count{route!="metrics",status_code=~"2.."}[1m]))',
    "qps_5xx":   'sum(rate(request_duration_seconds_count{route!="metrics",status_code=~"5.."}[1m]))',
    "frontend_p95": 'histogram_quantile(0.95, sum(rate(request_duration_seconds_bucket{name="front-end"}[1m])) by (le))',
}

print(f"Métricas configuradas: {list(METRICS.keys())}")

### 3a — Verificar conectividade com o Prometheus

Testa cada query no instante atual para garantir que o Prometheus está acessível e as queries retornam dados.

In [ ]:
now = now_epoch()
print(f"Timestamp atual: {utc_iso_from_epoch(now)}\n")

for name, promql in METRICS.items():
    val = query_prometheus_at(
        prom_url=PROM_URL, promql=promql, when_epoch=now, timeout=PROM_TIMEOUT_SECONDS
    )
    status = f"{val:.4f}" if val is not None else "None (sem dados ou erro)"
    print(f"  [{name}] → {status}")

---
## Etapa 4 — Execução das rodadas

Esta célula executa **todas** as rodadas sequencialmente (igual ao script). As sub-etapas estão desmembradas nas células seguintes para debug individual.

In [ ]:
# Estado compartilhado entre as células das sub-etapas
round_state: dict = {}
all_rows: List[Dict] = []

print("Estado inicializado. Execute as células de cada sub-etapa para rodar uma rodada por vez.")

### 4a — Iniciar uma rodada (registra timestamps e dispara o Locust)

Ajuste `CURRENT_ROUND` para a rodada que quer executar.

In [ ]:
CURRENT_ROUND = 1  # ← altere aqui para a rodada desejada (1, 2, 3…)

# ── Timestamps ───────────────────────────────────────────────────────────────
round_start = now_epoch()

if DISABLED_SECONDS > 0:
    print(f"Aguardando período DESATIVADA ({DISABLED_SECONDS}s)…")
    time.sleep(DISABLED_SECONDS)

warmup_start = now_epoch()
warmup_end   = warmup_start + WARMUP_SECONDS
run_end      = warmup_end + RUN_SECONDS
locust_total = WARMUP_SECONDS + RUN_SECONDS

round_state["round_number"] = CURRENT_ROUND
round_state["round_start"]  = round_start
round_state["warmup_start"] = warmup_start
round_state["warmup_end"]   = warmup_end
round_state["run_end"]      = run_end
round_state["locust_total"] = locust_total

print(f"Rodada {CURRENT_ROUND} iniciada")
print(f"  round_start  : {utc_iso_from_epoch(round_start)}")
print(f"  warmup_start : {utc_iso_from_epoch(warmup_start)}")
print(f"  warmup_end   : {utc_iso_from_epoch(warmup_end)}  (ready)")
print(f"  run_end      : {utc_iso_from_epoch(run_end)}")
print(f"  Duração total Locust: {locust_total}s")

### 4b — Disparar o Locust

> **Atenção:** esta célula bloqueia até o Locust terminar (warmup + execução = `locust_total` segundos).

In [ ]:
cmd = shlex.split(LOCUST_CMD) + [
    "-f", LOCUST_FILE,
    "--host", LOCUST_HOST,
    "--headless",
    "--users", str(USERS),
    "--spawn-rate", str(SPAWN_RATE),
    "--run-time", f"{round_state['locust_total']}s",
]
if LOCUST_EXTRA_ARGS:
    cmd.extend(LOCUST_EXTRA_ARGS.split())

print("Comando Locust:")
print(" ", " ".join(cmd))
print(f"\nAguardando {round_state['locust_total']}s…")

try:
    locust_result = subprocess.run(
        cmd,
        check=False,
        capture_output=True,
        text=True,
        timeout=round_state["locust_total"] + LOCUST_GRACE_SECONDS,
    )
    round_state["locust_result"] = locust_result
    round_state["round_end"] = now_epoch()
    print(f"\nLocust finalizado — exit code: {locust_result.returncode}")
    if locust_result.stdout:
        print("--- stdout ---")
        print(locust_result.stdout[-2000:])   # últimas 2000 chars para não poluir
    if locust_result.stderr:
        print("--- stderr ---")
        print(locust_result.stderr[-2000:])
except FileNotFoundError:
    raise RuntimeError(
        "Comando do Locust não encontrado. Ajuste LOCUST_CMD na Etapa 2."
    )
except subprocess.TimeoutExpired:
    print("[WARN] Locust excedeu o timeout — dados podem estar incompletos.")
    round_state["locust_result"] = None
    round_state["round_end"] = now_epoch()

if FAIL_ON_LOCUST_ERROR and round_state.get("locust_result") and round_state["locust_result"].returncode != 0:
    raise RuntimeError(
        f"Locust falhou na rodada {CURRENT_ROUND} com código {round_state['locust_result'].returncode}"
    )

### 4c — Coletar métricas no intervalo da rodada (range query)

In [ ]:
rs    = round_state["round_start"]
ws    = round_state["warmup_start"]
we    = round_state["warmup_end"]
re    = round_state["run_end"]
r_end = round_state.get("round_end", now_epoch())
lr    = round_state.get("locust_result")

data_end = min(max(re, ws), r_end)
round_state["data_end"] = data_end

print(f"Janela de coleta: {utc_iso_from_epoch(rs)} → {utc_iso_from_epoch(data_end)}")
print(f"Step: {SAMPLE_INTERVAL_SECONDS}s\n")

by_metric: Dict[str, Dict[int, Optional[float]]] = {}
for metric_name, promql in METRICS.items():
    print(f"Coletando [{metric_name}]…")
    by_metric[metric_name] = query_prometheus_range(
        prom_url=PROM_URL,
        promql=promql,
        start_epoch=rs,
        end_epoch=data_end,
        step_seconds=SAMPLE_INTERVAL_SECONDS,
        timeout=PROM_TIMEOUT_SECONDS,
    )
    non_null = sum(1 for v in by_metric[metric_name].values() if v is not None)
    print(f"  → {len(by_metric[metric_name])} pontos, {non_null} com valor")

round_state["by_metric"] = by_metric
print("\nColeta de range concluída.")

### 4d — Construir linhas do CSV para esta rodada

In [ ]:
rs       = round_state["round_start"]
ws       = round_state["warmup_start"]
we       = round_state["warmup_end"]
re       = round_state["run_end"]
data_end = round_state["data_end"]
lr       = round_state.get("locust_result")
by_metric = round_state["by_metric"]
metric_names = list(METRICS.keys())
exit_code = lr.returncode if lr else -1

round_rows: List[Dict] = []

ts_points = list(range(int(rs), int(data_end) + 1, SAMPLE_INTERVAL_SECONDS))
for ts in ts_points:
    row: Dict = {
        "timestamp_utc":   utc_iso_from_epoch(ts),
        "round":           round_state["round_number"],
        "load_state":      state_for_timestamp(ts, ws, we, re),
        "round_start_utc": utc_iso_from_epoch(rs),
        "warmup_start_utc":utc_iso_from_epoch(ws),
        "ready_utc":       utc_iso_from_epoch(we),
        "round_end_utc":   utc_iso_from_epoch(data_end),
        "locust_exit_code":exit_code,
    }
    for mn in metric_names:
        row[mn] = by_metric.get(mn, {}).get(ts)
    round_rows.append(row)

# Linha especial do ponto PRONTA (snapshot no fim do warmup)
ready_row: Dict = {
    "timestamp_utc":   utc_iso_from_epoch(we),
    "round":           round_state["round_number"],
    "load_state":      "PRONTA",
    "round_start_utc": utc_iso_from_epoch(rs),
    "warmup_start_utc":utc_iso_from_epoch(ws),
    "ready_utc":       utc_iso_from_epoch(we),
    "round_end_utc":   utc_iso_from_epoch(data_end),
    "locust_exit_code":exit_code,
}
for mn, promql in METRICS.items():
    ready_row[mn] = query_prometheus_at(
        prom_url=PROM_URL, promql=promql, when_epoch=we, timeout=PROM_TIMEOUT_SECONDS
    )

round_rows.append(ready_row)
all_rows.extend(round_rows)

print(f"Rodada {round_state['round_number']}: {len(round_rows)} linhas geradas")
print(f"Total acumulado: {len(all_rows)} linhas")

# Preview
print("\nPrimeiras linhas da rodada:")
for r in round_rows[:3]:
    print(" ", {k: v for k, v in r.items()})

> Para executar mais rodadas, volte à **Etapa 4a**, altere `CURRENT_ROUND` e execute as células 4a → 4d novamente.

---
## Etapa 5 — Salvar CSV

In [ ]:
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)

headers = [
    "timestamp_utc", "round", "load_state",
    "round_start_utc", "warmup_start_utc", "ready_utc", "round_end_utc",
    "locust_exit_code",
    *list(METRICS.keys()),
]

with out_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for row in all_rows:
        writer.writerow(row)

print(f"CSV salvo em: {out_path.resolve()}")
print(f"Total de linhas: {len(all_rows)}")

---
## Etapa 6 — Análise rápida dos resultados

Carrega o CSV gerado e exibe estatísticas e gráficos básicos para inspeção imediata.

In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("pandas/matplotlib não instalados — instale com: pip install pandas matplotlib")

if HAS_PANDAS:
    df = pd.read_csv(OUTPUT_CSV, parse_dates=["timestamp_utc"])
    print(f"Shape: {df.shape}")
    print(f"load_state únicos: {df['load_state'].unique()}")
    display(df.head(10))

In [ ]:
if HAS_PANDAS:
    print("Estatísticas por load_state:")
    metric_cols = list(METRICS.keys())
    display(df.groupby("load_state")[metric_cols].describe().round(4))

In [ ]:
if HAS_PANDAS:
    fig, axes = plt.subplots(len(METRICS), 1, figsize=(14, 4 * len(METRICS)), sharex=True)
    if len(METRICS) == 1:
        axes = [axes]

    colors = {"DESATIVADA": "gray", "AQUECIMENTO": "orange", "EXECUCAO": "steelblue", "PRONTA": "green"}

    for ax, (metric_name, _) in zip(axes, METRICS.items()):
        for state, grp in df.groupby("load_state"):
            ax.scatter(
                grp["timestamp_utc"], grp[metric_name],
                label=state, color=colors.get(state, "black"), s=20, alpha=0.7
            )
        ax.set_ylabel(metric_name)
        ax.legend(loc="upper right", fontsize=7)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("timestamp_utc")
    fig.suptitle("Métricas por fase — todas as rodadas", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Etapa 7 — (Opcional) Análise do CSV existente

Use esta célula para inspecionar um CSV já coletado sem precisar reexecutar o experimento.

In [ ]:
EXISTING_CSV = "../data/resultado_experimento_1.csv"  # ← altere se necessário

if HAS_PANDAS:
    df_existing = pd.read_csv(EXISTING_CSV, parse_dates=["timestamp_utc"])
    print(f"Shape: {df_existing.shape}")
    display(df_existing.head())
    print("\nEstatísticas das métricas (EXECUCAO):")
    exec_df = df_existing[df_existing["load_state"] == "EXECUCAO"]
    metric_cols_existing = [c for c in df_existing.columns if c not in [
        "timestamp_utc", "round", "load_state",
        "round_start_utc", "warmup_start_utc", "ready_utc", "round_end_utc", "locust_exit_code"
    ]]
    display(exec_df[metric_cols_existing].describe().round(4))